In [85]:
import wandb
import pandas as pd
import os
from tqdm.notebook import tqdm

# from table_plotter import print_result_table

In [86]:
api = wandb.Api(timeout=600)


In [87]:
# Specify cache directory
cache_dir = "./wandb_cache"
os.makedirs(cache_dir, exist_ok=True)

In [88]:

skipped_runs = []  # List to store IDs of skipped runs

evaluation_keys = ['Evaluation/acc_imp_perc', 'Evaluation/exist_imp_perc', 'Evaluation/reach_imp_perc', 'Evaluation/path_length',
                   'Evaluation/fn_imp_perc', 'Evaluation/fp_imp_perc', 'Evaluation/tn_imp_perc', 'Evaluation/tp_imp_perc', 
                   'Evaluation/solvability', 'Evaluation/playability']
evaluation2_keys = ['Evaluation/playability', 'Evaluation/naive_playability', 'Evaluation/solvability', 'Evaluation/acc_imp_perc']


In [89]:
def get_dataframe_from_run(run):
    dfs = list()
    
    for run in tqdm(runs):
    
        # Define cache filename based on run ID
        cache_file = os.path.join(cache_dir, f"{run.id}.csv")
        
        # Check if cached file exists
        if os.path.exists(cache_file):
            # Load cached DataFrame
            df = pd.read_csv(cache_file)
        else:
            if run.state == "running":
                print(f"Skipping run ID: {run.id} (state: {run.state})")
                continue
            
            df = run.history(keys=["Evaluation/llm_iteration", *evaluation_keys[:1]])
    
            def append_key(src_df, key):
    
                tgt_df = run.history(keys=[key, "Evaluation/llm_iteration"])
                src_df = pd.merge(src_df, tgt_df, on="Evaluation/llm_iteration", how="outer")
                src_df = src_df.drop(columns=["_step_x", "_step_y"], errors="ignore")
                return src_df
    
            for key in evaluation_keys[1:]:
                try:
                    df = append_key(df, key)
                except Exception as e:
                    print(f"Error: {e} at run ID: {run.id}")
    
            
            # Add run config to DataFrame with prefix 'config.'
            for key, value in run.config.items():
                if isinstance(value, list):
                    value = ",".join(map(str, value))  # Convert list to comma-separated string
                df[key] = value
    
            # 기본값 설정
            default_values = {'n_self_alignment': 0, 'feedback_type': 'default'}
            # 열이 없을 경우 기본값으로 채워 넣기
            for col, value in default_values.items():
                if col not in df.columns:
                    df[col] = value
            
             
            # Filter columns
            key_filter = ['run_id', 'final_state', 'target_character', 'pe', 'gpt_model', 'branch_factor', 'exp_name', 'evaluator', 'total_iterations', 'n_self_alignment', 'feedback_type', 'feedback_input_type', 'total_timesteps', 
                          'reward_feature', 'fewshot', 'problem', 'seed', 
                          'Evaluation/llm_iteration'] + evaluation_keys
            auxiliary_key_filter = []
            
            df['run_id'] = run.id  # Add run ID as a column
            df['final_state'] = run.state
            
            try:
                df = df[key_filter + auxiliary_key_filter]
            except KeyError:
                df = df[key_filter]
            
            # Save DataFrame to cache as CSV
            df.to_csv(cache_file, index=False)
        
        dfs.append(df)
    
    # Concatenate all DataFrames
    df = pd.concat(dfs, ignore_index=True)
    
    return df

In [90]:
def get_dataframe_from_run2(run):
    dfs = []
    
    for run in tqdm(runs):
    
        # Define cache filename based on run ID
        cache_file = os.path.join(cache_dir, f"{run.id}.csv")
        
        # Check if cached file exists
        if os.path.exists(cache_file):
            # Load cached DataFrame
            df = pd.read_csv(cache_file)
        else:
            if run.state == "running":
                print(f"Skipping run ID: {run.id} (state: {run.state})")
                continue
            
            df = run.history(keys=["Evaluation/llm_iteration", *evaluation2_keys[:1]])
    
            def append_key(src_df, key):
    
                tgt_df = run.history(keys=[key, "Evaluation/llm_iteration"])
                src_df = pd.merge(src_df, tgt_df, on="Evaluation/llm_iteration", how="outer")
                src_df = src_df.drop(columns=["_step_x", "_step_y"], errors="ignore")
                return src_df
    
            for key in evaluation2_keys[1:]:
                try:
                    df = append_key(df, key)
                except Exception as e:
                    print(f"Error: {e} at run ID: {run.id}")
    
            
            # Add run config to DataFrame with prefix 'config.'
            for key, value in run.config.items():
                if isinstance(value, list):
                    value = ",".join(map(str, value))  # Convert list to comma-separated string
                df[key] = value
    
            # 기본값 설정
            default_values = {'n_self_alignment': 0, 'feedback_type': 'default'}
            # 열이 없을 경우 기본값으로 채워 넣기
            for col, value in default_values.items():
                if col not in df.columns:
                    df[col] = value
            
             
            # Filter columns
            key_filter = ['run_id', 'final_state', 'target_character', 'pe', 'gpt_model', 'branch_factor', 'exp_name', 'evaluator', 'total_iterations', 'n_self_alignment', 'feedback_type', 'feedback_input_type', 'total_timesteps', 
                          'reward_feature', 'fewshot', 'problem', 'seed', 
                          'Evaluation/llm_iteration'] + evaluation2_keys
            auxiliary_key_filter = []
            
            df['run_id'] = run.id  # Add run ID as a column
            df['final_state'] = run.state
            
            try:
                df = df[key_filter + auxiliary_key_filter]
            except KeyError:
                df = df[key_filter]
            
            # Save DataFrame to cache as CSV
            df.to_csv(cache_file, index=False)
        
        dfs.append(df)
    
    # Concatenate all DataFrames
    df = pd.concat(dfs, ignore_index=True)
    
    return df

In [91]:
runs = api.runs("inchangbaek4907/scenario-feedback")
scenario_df = get_dataframe_from_run(runs)
scenario_df = scenario_df[scenario_df['Evaluation/llm_iteration'] <= 6].copy()
# set score column with acc_imp_perc
scenario_df['score'] = scenario_df['Evaluation/acc_imp_perc']
scenario_df

  0%|          | 0/60 [00:00<?, ?it/s]

,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/exist_imp_perc,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score
0,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.0,0.966667,26.344828,0.1,1.933333,0.0,0.966667,0.966667,0.966667,0.322222
1,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.000002,0.0,2.000000,0.0,1.000000,1.000000,1.000000,0.333333
2,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.000002,0.0,2.000000,0.0,1.000000,1.000000,1.000000,0.333333
3,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.066668,0.0,2.000000,0.0,1.000000,1.000000,1.000000,0.333333
4,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.000002,0.0,2.000000,0.0,1.000000,1.000000,1.000000,0.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,2d096q4m,finished,2,got,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.266668,0.0,1.000000,0.0,2.000000,1.000000,1.000000,0.666667
356,2d096q4m,finished,2,got,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.066668,0.0,1.000000,0.0,2.000000,0.966667,1.000000,0.666667
357,2d096q4m,finished,2,got,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.066668,0.0,1.000000,0.0,2.000000,0.966667,1.000000,0.666667
358,2d096q4m,finished,2,got,gpt-4o,2,feedback,hr,6,5,...,1.0,1.000000,26.214287,0.2,0.933333,0.0,1.866667,0.933333,0.933333,0.622222


In [92]:
runs = api.runs("inchangbaek4907/scenario2-feedback")
scenario2_df = get_dataframe_from_run2(runs).copy()
scenario2_df = scenario2_df[scenario2_df['Evaluation/llm_iteration'] <= 6].copy()
# set score column with playability
scenario2_df['score'] = scenario2_df['Evaluation/playability']
scenario2_df

  0%|          | 0/60 [00:00<?, ?it/s]

,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,reward_feature,fewshot,problem,seed,Evaluation/llm_iteration,Evaluation/playability,Evaluation/naive_playability,Evaluation/solvability,Evaluation/acc_imp_perc,score
0,94q5bedo,finished,5,got,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,0,1,0.0,0.000000,0.000000,0.0,0.0
1,94q5bedo,finished,5,got,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,0,2,0.0,0.000000,0.000000,0.0,0.0
2,94q5bedo,finished,5,got,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,0,3,0.0,0.000000,0.000000,0.0,0.0
3,94q5bedo,finished,5,got,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,0,4,0.0,0.000000,0.000000,0.0,0.0
4,94q5bedo,finished,5,got,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,0,5,0.0,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,szkn8zgd,finished,6,tot,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,10,2,0.0,0.633333,0.633333,1.0,0.0
356,szkn8zgd,finished,6,tot,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,10,3,0.0,0.700000,0.700000,1.0,0.0
357,szkn8zgd,finished,6,tot,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,10,4,0.0,1.000000,1.000000,1.0,0.0
358,szkn8zgd,finished,6,tot,gpt-4o,2,feedback,hr,6,5,...,array,False,dungeon4,10,5,0.0,0.633333,0.633333,1.0,0.0


In [93]:
scenario_df = pd.concat([scenario_df, scenario2_df], ignore_index=True)
scenario_df

,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score,Evaluation/naive_playability
0,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,0.966667,26.344828,0.1,1.933333,0.0,0.966667,0.966667,0.966667,0.322222,NaN
1,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.000000,26.000002,0.0,2.000000,0.0,1.000000,1.000000,1.000000,0.333333,NaN
2,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.000000,26.000002,0.0,2.000000,0.0,1.000000,1.000000,1.000000,0.333333,NaN
3,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.000000,26.066668,0.0,2.000000,0.0,1.000000,1.000000,1.000000,0.333333,NaN
4,qdy8rirf,finished,1,tot,gpt-4o,2,feedback,hr,6,5,...,1.000000,26.000002,0.0,2.000000,0.0,1.000000,1.000000,1.000000,0.333333,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,szkn8zgd,finished,6,tot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.633333,0.000000,0.000000,0.633333
716,szkn8zgd,finished,6,tot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.700000,0.000000,0.000000,0.700000
717,szkn8zgd,finished,6,tot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000,0.000000,1.000000
718,szkn8zgd,finished,6,tot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.633333,0.000000,0.000000,0.633333


In [94]:
# Print summary of skipped runs
print("\nSummary of Skipped Runs:")
print(f"Total skipped runs: {len(skipped_runs)}")
print("Skipped run IDs:", skipped_runs)


Summary of Skipped Runs:
Total skipped runs: 0
Skipped run IDs: []


In [95]:
df = pd.concat([scenario_df], ignore_index=True)

In [96]:
df.to_csv(f"feedback_result.csv", index=False)